# 第4讲：统计推断与线性回归（学生操练）

        > 课程：《交通大数据分析与应用》  
        > 数据：课程模拟数据，不是实际监测数据  
        > 建议用时：20分钟

        ## 目标

        1. 修改CONFIDENCE_LEVEL并比较区间宽度；
2. 按时间先后划分训练集与检验集，计算MAE和R²；
3. 选择一个系数，写出包含单位和控制条件的解释；

        代码可以直接运行；请按`TODO`修改参数、核对输出并完成解释。

## 1. Setup｜环境、路径与参数

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25})
RANDOM_STATE = 42
print("环境已就绪；数据目录：", DATA_DIR.resolve())

## 2. 计算总体平均速度的区间估计

In [ ]:
from scipy import stats
traffic = pd.read_csv(DATA_DIR / "traffic_15min.csv", parse_dates=["timestamp"])
sample = traffic.loc[(traffic["detector_id"] == "D03") & (traffic["is_weekend"] == 0), "speed_kmh"].dropna()
CONFIDENCE_LEVEL = 0.95  # TODO：改为0.90或0.99
mean_speed = sample.mean(); se = stats.sem(sample)
interval = stats.t.interval(CONFIDENCE_LEVEL, df=len(sample)-1, loc=mean_speed, scale=se)
print(f"n={len(sample)}, mean={mean_speed:.2f} km/h, {CONFIDENCE_LEVEL:.0%} CI=({interval[0]:.2f}, {interval[1]:.2f})")

## 3. 按时间划分并拟合线性回归

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
features = ["flow_15min", "occupancy_pct", "rain", "lanes"]
model_data = traffic.sort_values("timestamp").dropna(subset=features + ["speed_kmh"])
cut = int(len(model_data) * 0.75)
train, test = model_data.iloc[:cut], model_data.iloc[cut:]
model = LinearRegression().fit(train[features], train["speed_kmh"])
pred = model.predict(test[features])
coef_table = pd.DataFrame({"feature": features, "coefficient": model.coef_})
display(coef_table.round(3))
print("MAE", round(mean_absolute_error(test["speed_kmh"], pred), 3), "R2", round(r2_score(test["speed_kmh"], pred), 3))

## 4. 检查残差结构

In [ ]:
residual = test["speed_kmh"].to_numpy() - pred
plt.scatter(pred, residual, s=9, alpha=.35, color="#2F5597")
plt.axhline(0, color="#C00000", linewidth=1)
plt.xlabel("Fitted speed (km/h)"); plt.ylabel("Residual (km/h)"); plt.title("Residual check")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "lesson04_residuals.png"); plt.show()
coef_table.to_csv(OUTPUT_DIR / "lesson04_coefficients.csv", index=False)

## 5. 完成自检

In [ ]:
checks = {"训练早于检验": train["timestamp"].max() <= test["timestamp"].min(), "区间包含样本均值": interval[0] < mean_speed < interval[1], "残差图已生成": (OUTPUT_DIR / "lesson04_residuals.png").exists()}
status = "PASS" if all(checks.values()) else "CHECK"
(OUTPUT_DIR / "自检结果.txt").write_text(status, encoding="utf-8")
print(status, checks)

## Checks｜当堂记录

        - 修改CONFIDENCE_LEVEL并比较区间宽度
- 按时间先后划分训练集与检验集，计算MAE和R²
- 选择一个系数，写出包含单位和控制条件的解释

        **预期结果：** 均值置信区间、回归系数表、残差图与样本外误差。

        **完成标准：** 区间对象不是单辆车；系数解释包含单位和其他变量不变；Notebook生成PASS。

        请在课堂记录中写下：改了什么参数、结果发生了什么变化、这个变化在交通问题中意味着什么。